# Gradient descent

The normal equations solve the least-squares problem in one linear-algebra step. Gradient descent solves the same problem by walking downhill on the loss. On a small table the first steps can be written by hand, and they land on the same line if the step length is safe.


## The step

Use the loss with a one-half, so the factor $2$ from the square does not appear in the gradient:

$$
L(\beta) = \dfrac{1}{2}\|y - X\beta\|^2.
$$

Its gradient is

$$
\nabla L(\beta) = -X^{\mathsf T}(y - X\beta) = -X^{\mathsf T}e.
$$

A step of length $\alpha$ moves against the gradient,

$$
\beta_{\mathrm{new}} = \beta + \alpha\, X^{\mathsf T}e.
$$

At a least-squares solution, $X^{\mathsf T}e = 0$, so the walk stops. Anywhere else it moves.


## Two exact steps on the three-point table

Start at $\beta = (0, 0)$, with $X$ the usual three-point design and $y = (2, 3, 5)^{\mathsf T}$. Then $e = y$ and

$$
X^{\mathsf T}e = \begin{bmatrix} 10 \\ 23 \end{bmatrix}.
$$

The loss is $L = \tfrac{1}{2}(4 + 9 + 25) = 19$, so the residual sum of squares is $38$. Take $\alpha = 1/20$:

$$
\beta^{(1)} = \dfrac{1}{20}\begin{bmatrix} 10 \\ 23 \end{bmatrix} = \begin{bmatrix} 1/2 \\ 23/20 \end{bmatrix}.
$$

The new residuals are

$$
e = \dfrac{7}{20},\ \dfrac{1}{5},\ \dfrac{21}{20}, \qquad
X^{\mathsf T}e = \begin{bmatrix} 8/5 \\ 39/10 \end{bmatrix}, \qquad
\mathrm{RSS} = \dfrac{253}{200}.
$$

One more step:

$$
\beta^{(2)}
= \begin{bmatrix} 1/2 \\ 23/20 \end{bmatrix}
+ \dfrac{1}{20}\begin{bmatrix} 8/5 \\ 39/10 \end{bmatrix}
= \begin{bmatrix} 29/50 \\ 269/200 \end{bmatrix}.
$$

The walk has moved from $(0, 0)$ toward the known solution $(1/3,\ 3/2) \approx (0.333,\ 1.5)$. It has not arrived after two steps. Descent is a sequence, not a closed form. The slow direction of $X^{\mathsf T}X$ shrinks by only about $0.982$ per step when $\alpha = 1/20$, so a couple of thousand steps are what make the walk meet the normal-equation solution.


In [1]:
# Two hand steps of gradient descent, then a longer walk to the normal-equation fit.
from fractions import Fraction
import numpy as np
X = np.array([[1.0, 1], [1, 2], [1, 3]])
y = np.array([2.0, 3, 5])
beta = np.zeros(2)
alpha = 0.05
first = beta + alpha * (X.T @ y)
residual = y - X @ first
second = first + alpha * (X.T @ residual)
print("step 1", first, "RSS", np.sum((y - X @ first) ** 2))
print("step 2", second)
assert np.allclose(first, [1 / 2, 23 / 20])
assert abs(np.sum((y - X @ first) ** 2) - 253 / 200) < 1e-12
assert np.allclose(second, [29 / 50, 269 / 200])
beta = np.zeros(2)
for _ in range(2000):
    beta = beta + alpha * (X.T @ (y - X @ beta))
print("after 2000 steps", beta)
assert np.allclose(beta, [1 / 3, 3 / 2], atol=1e-8)


step 1 [0.5  1.15] RSS 1.2649999999999995
step 2 [0.58  1.345]
after 2000 steps [0.33333333 1.5       ]


## One change: a longer step

The same first gradient with $\alpha = 1/10$ lands on

$$
\beta = \begin{bmatrix} 1 \\ 23/10 \end{bmatrix}.
$$

The residual sum of squares is

$$
\left(-\dfrac{13}{10}\right)^2 + \left(-\dfrac{13}{5}\right)^2 + \left(-\dfrac{29}{10}\right)^2 = \dfrac{843}{50} = 16.86.
$$

That is better than the starting value $38$ and much worse than $253/200 = 1.265$, the cost after the shorter step. A larger step is not automatically faster. This one jumped past the valley floor.


In [2]:
# Doubling the step length overshoots and leaves a larger residual sum of squares.
import numpy as np
X = np.array([[1.0, 1], [1, 2], [1, 3]])
y = np.array([2.0, 3, 5])
beta = 0.1 * (X.T @ y)
rss = np.sum((y - X @ beta) ** 2)
print("beta", beta)
print("RSS", rss)
assert np.allclose(beta, [1, 2.3])
assert abs(rss - 843 / 50) < 1e-12
assert rss > 253 / 200


beta [1.  2.3]
RSS 16.860000000000007


## Pitfall

With $\alpha = 1$ the first step goes to $(10, 23)$. The predictions are enormous relative to responses $2, 3, 5$, and the residual sum of squares is $9246$, far above the starting cost $38$. The step length has to be small compared with the reciprocal of the largest eigenvalue of $X^{\mathsf T}X$. Here that matrix is $\bigl[\begin{smallmatrix} 3 & 6 \\ 6 & 14 \end{smallmatrix}\bigr]$, whose larger eigenvalue is $(17 + \sqrt{265})/2 \approx 16.64$. A constant step longer than about $2/16.64 \approx 0.12$ is not a guaranteed descent method for this quadratic. $\alpha = 1$ is well beyond that bound.


In [3]:
# A step of length 1 increases the residual sum of squares.
import numpy as np
X = np.array([[1.0, 1], [1, 2], [1, 3]])
y = np.array([2.0, 3, 5])
beta = X.T @ y
rss = np.sum((y - X @ beta) ** 2)
gram = X.T @ X
eigenvalues = np.linalg.eigvalsh(gram)
print("beta", beta)
print("RSS", rss)
print("eigenvalues of X'X", eigenvalues)
assert np.allclose(beta, [10, 23])
assert abs(rss - 9246) < 1e-8
assert rss > 38
assert eigenvalues.max() > 16


beta [10. 23.]
RSS 9246.0
eigenvalues of X'X [ 0.3605897 16.6394103]


## Summary

- The gradient of $\tfrac{1}{2}\|y-X\beta\|^2$ is $-X^{\mathsf T}e$. The update adds $\alpha X^{\mathsf T}e$.
- From the origin with $\alpha = 1/20$, the first two coefficients are $(1/2,\ 23/20)$ and $(29/50,\ 269/200)$.
- $\alpha = 1/10$ overshoots, and $\alpha = 1$ increases the cost. Two thousand safe steps reach $(1/3,\ 3/2)$.
